# 02 - SEC EDGAR Filings Ingestion (MongoDB)

**Project:** SEC Event Study Analytics System  
**Author contribution:** SEC EDGAR REST API pipeline, MongoDB schema design, raw text extraction

## What this notebook does
1. Connects to SEC EDGAR REST API to fetch 10-K, 10-Q, 8-K filing metadata per company
2. Downloads raw filing HTML and extracts clean text via BeautifulSoup
3. Stores 65,000+ documents in MongoDB `sec_filings` collection (3.85 GB)

## MongoDB Schema: `sec_filings`
| Field | Type | Notes |
|---|---|---|
| `cik` + `accession_number` | Compound unique key | Prevents duplicate ingestion |
| `raw_text` | String | Full filing text, enables keyword search |
| `filing_type` | String | 10-K / 10-Q / 8-K |
| `filing_date` | String (ISO) | Parsed from SEC metadata |

## Setup
```bash
# Set SEC User-Agent in .env:
# SEC_USER_AGENT=Your Name your@email.com
```

> **Note:** SEC EDGAR requires a valid `User-Agent` header with your name and email.  
> Update the `SEC_API_HEADERS` dict with your own credentials before running.

## Extensibility
- Change `FIXED_START` date to extend/shorten the ingestion window
- Add more filing types (e.g. `DEF 14A`, `S-1`) to the filter list in `extract_filings_meta()`


# Create sec_fillings table & insert data from SEC EDGAR

In [1]:
!pip install -U "psycopg[binary]" pymongo requests beautifulsoup4 pandas


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import psycopg
from pymongo import MongoClient
import requests
from bs4 import BeautifulSoup
from datetime import date
import time

pg_conn = psycopg.connect(
    host=os.environ.get("PG_HOST", "localhost"),
    port=os.environ.get("PG_PORT", "5432"),
    dbname=os.environ.get("PG_DBNAME", "final"), 
    user=os.environ.get("PG_USER", "postgres"),
    password=os.environ.get("PG_PASSWORD", "your_password")
)
pg_cur = pg_conn.cursor()
print("Connected to PostgreSQL 'final'")

Connected to PostgreSQL 'final'


In [3]:
pg_cur.execute("""
    SELECT company_id, cik, ticker, company_name
    FROM company
    ORDER BY company_id;
""")

rows = pg_cur.fetchall()
print("Total rows fetched:", len(rows))

companies = []
for row in rows:
    company_id, cik, ticker, company_name = row
    companies.append({
        "company_id": company_id,
        "cik": cik,
        "ticker": ticker,
        "company_name": company_name
    })

for c in companies[:5]:
    print(c)

Total rows fetched: 927
{'company_id': 1, 'cik': '0000002186', 'ticker': 'BKTI', 'company_name': 'BK Technologies Corp'}
{'company_id': 2, 'cik': '0000002488', 'ticker': 'AMD', 'company_name': 'ADVANCED MICRO DEVICES INC'}
{'company_id': 3, 'cik': '0000004127', 'ticker': 'SWKS', 'company_name': 'SKYWORKS SOLUTIONS, INC.'}
{'company_id': 4, 'cik': '0000006281', 'ticker': 'ADI', 'company_name': 'ANALOG DEVICES INC'}
{'company_id': 5, 'cik': '0000006951', 'ticker': 'AMAT', 'company_name': 'APPLIED MATERIALS INC /DE'}


In [2]:
import os
client = MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))
db = client["sec_project"]
sec_filings = db["sec_filings"]

print("Connected to MongoDB: DB='sec_project', collection='sec_filings'")

Connected to MongoDB: DB='sec_project', collection='sec_filings'


In [5]:
db.drop_collection("sec_filings")
sec_filings = db["sec_filings"]

sec_filings.create_index([("cik", 1), ("accession_number", 1)], unique=True)
sec_filings.create_index([("company_id", 1), ("filing_type", 1), ("filing_date", -1)])
sec_filings.create_index([("raw_text", "text")])

print("✅ Recreated collection 'sec_filings'")
print("Collection reset. Current count:", sec_filings.count_documents({}))

✅ Recreated collection 'sec_filings'
Collection reset. Current count: 0


In [6]:
import os
SEC_API_HEADERS = {
    "User-Agent": "your_name your_email@example.com  # Replace with your info",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

SEC_ARCHIVES_HEADERS = {
    "User-Agent": "your_name your_email@example.com  # Replace with your info",
    "Accept-Encoding": "gzip, deflate"
}

FIXED_START = date(2015, 11, 28)

def fetch_company_submissions(cik_10):
    url = f"https://data.sec.gov/submissions/CIK{cik_10}.json"
    resp = requests.get(url, headers=SEC_API_HEADERS)
    if resp.status_code != 200:
        print(f"❌ Failed submissions for {cik_10}, status={resp.status_code}")
        return None
    return resp.json()

def extract_filings_meta(sub_json, cik_10):
    if not sub_json:
        return []

    recent = sub_json["filings"]["recent"]
    forms = recent.get("form", [])
    dates = recent.get("filingDate", [])
    accs  = recent.get("accessionNumber", [])
    docs  = recent.get("primaryDocument", [])

    filings = []
    for form, d, acc, doc in zip(forms, dates, accs, docs):
        if form not in ("10-K", "10-Q","8-K"):
            continue
        try:
            y, m, da = map(int, d.split("-"))
            f_date = date(y, m, da)
        except:
            continue
        if f_date < FIXED_START:
            continue

        filings.append({
            "cik": cik_10,
            "filing_type": form,
            "filing_date": f_date,
            "accession_number": acc,
            "primary_doc": doc
        })

    return filings

In [7]:
def build_filing_paths(cik_10, accession_number, primary_doc):
    cik_no_zero = str(int(cik_10))
    acc_nodash = accession_number.replace("-", "")

    base_dir = f"https://www.sec.gov/Archives/edgar/data/{cik_no_zero}/{acc_nodash}/"
    index_url = base_dir + f"{accession_number}-index.htm"
    main_doc_url = base_dir + primary_doc  # e.g. bkti20250930_10q.htm

    return base_dir, index_url, main_doc_url

def download_html(url):
    resp = requests.get(url, headers=SEC_ARCHIVES_HEADERS)
    if resp.status_code != 200:
        print(f"  ❌ Failed: {url} status={resp.status_code}")
        return None
    return resp.text

def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for s in soup(["script", "style"]):
        s.decompose()
    lines = [line.strip() for line in soup.get_text("\n").splitlines()]
    return "\n".join(l for l in lines if l)


In [8]:
def process_company_filings(company):
    company_id = company["company_id"]
    cik_10 = company["cik"].zfill(10)
    ticker = company["ticker"]
    name = company["company_name"]

    print(f"\n=== CIK={cik_10} ticker={ticker} ===")

    sub_json = fetch_company_submissions(cik_10)
    if sub_json is None:
        print("  ⚠️ No submissions JSON")
        return

    filings = extract_filings_meta(sub_json, cik_10)
    print(f"  Found {len(filings)} filings.")

    inserted = 0

    for meta in filings:
        base_dir, index_url, main_doc_url = build_filing_paths(
            cik_10,
            meta["accession_number"],
            meta["primary_doc"]
        )

        print(f"  -> {meta['filing_type']} {meta['filing_date']}")

        # Try main doc first
        html = download_html(main_doc_url)
        used_index = False

        if html is None:
            print("     ⚠️ main doc failed; trying index")
            html = download_html(index_url)
            if html is None:
                print("     ⛔ Skip filing")
                continue
            used_index = True

        text = html_to_text(html)

        doc = {
            "company_id": company_id,
            "cik": cik_10,
            "ticker": ticker,
            "company_name": name,

            "filing_type": meta["filing_type"],
            "filing_date": meta["filing_date"].isoformat(),
            "accession_number": meta["accession_number"],
            "primary_doc": meta["primary_doc"],

            "source_main": main_doc_url,
            "source_index": index_url,
            "used_index_page": used_index,

            "raw_text": text
        }

        try:
            sec_filings.insert_one(doc)
            inserted += 1
            print("     ✔ inserted")
        except Exception as e:
            print("     ⚠️ skipped:", e)

    print(f"  ✅ Done. Inserted {inserted}.")


In [9]:
print("Total companies:", len(companies))

for idx, company in enumerate(companies, start=1):
    print(f"\n=== [{idx}/{len(companies)}] {company['ticker']} ===")
    process_company_filings(company)
    time.sleep(0.5)  

print("\n🎉 All companies processed!")

Total companies: 927

=== [1/927] BKTI ===

=== CIK=0000002186 ticker=BKTI ===
  Found 183 filings.
  -> 10-Q 2025-11-06
     ✔ inserted
  -> 8-K 2025-11-06
     ✔ inserted
  -> 8-K 2025-10-30
     ✔ inserted
  -> 8-K 2025-10-06
     ✔ inserted
  -> 10-Q 2025-08-14
     ✔ inserted
  -> 8-K 2025-08-14
     ✔ inserted
  -> 8-K 2025-07-14
     ✔ inserted
  -> 8-K 2025-06-18
     ✔ inserted
  -> 8-K 2025-06-03
     ✔ inserted
  -> 8-K 2025-05-13
     ✔ inserted
  -> 10-Q 2025-05-13
     ✔ inserted
  -> 8-K 2025-03-27
     ✔ inserted
  -> 10-K 2025-03-27
     ✔ inserted
  -> 8-K 2024-11-14
     ✔ inserted
  -> 10-Q 2024-11-14
     ✔ inserted
  -> 8-K 2024-11-06
     ✔ inserted
  -> 8-K 2024-11-04
     ✔ inserted
  -> 8-K 2024-08-08
     ✔ inserted
  -> 10-Q 2024-08-08
     ✔ inserted
  -> 8-K 2024-06-20
     ✔ inserted
  -> 8-K 2024-05-13
     ✔ inserted
  -> 8-K 2024-05-09
     ✔ inserted
  -> 10-Q 2024-05-09
     ✔ inserted
  -> 8-K 2024-03-14
     ✔ inserted
  -> 10-K 2024-03-14
     ✔ i

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



     ✔ inserted
  -> 8-K 2023-11-13
     ✔ inserted
  -> 10-Q 2023-11-08
     ✔ inserted
  -> 8-K 2023-11-07
     ✔ inserted
  -> 10-Q 2023-08-09
     ✔ inserted
  -> 8-K 2023-08-08
     ✔ inserted
  -> 8-K 2023-06-20
     ✔ inserted
  -> 10-Q 2023-05-10
     ✔ inserted
  -> 8-K 2023-05-09
     ✔ inserted
  -> 8-K 2023-04-24
     ✔ inserted
  -> 8-K 2023-03-09
     ✔ inserted
  -> 10-K 2023-03-01
     ✔ inserted
  -> 8-K 2023-02-28
     ✔ inserted
  -> 10-Q 2022-11-10
     ✔ inserted
  -> 8-K 2022-11-09
     ✔ inserted
  -> 8-K 2022-09-28
     ✔ inserted
  -> 10-Q 2022-08-11
     ✔ inserted
  -> 8-K 2022-08-10
     ✔ inserted
  -> 8-K 2022-07-15
     ✔ inserted
  -> 8-K 2021-06-07
     ✔ inserted
  -> 8-K 2021-06-03
     ✔ inserted
  -> 8-K 2021-05-04
     ✔ inserted
  -> 8-K 2021-05-03
     ✔ inserted
  -> 8-K 2021-04-21
     ✔ inserted
  ✅ Done. Inserted 84.

=== [715/927] KVYO ===

=== CIK=0001835830 ticker=KVYO ===
  Found 27 filings.
  -> 10-Q 2025-11-05
     ✔ inserted
  -> 8-K 2

In [10]:
print("db name:", db.name)
print("collection full name:", sec_filings.full_name)
print("docs:", sec_filings.count_documents({}))

db name: sec_project
collection full name: sec_project.sec_filings
docs: 65440


In [11]:
print("Total filings stored:", sec_filings.count_documents({}))

sample = sec_filings.find_one(sort=[("_id", -1)])
if sample:
    print("\nSample doc:", sample["ticker"], sample["filing_type"],
          sample["filing_date"], "| used_index =", sample["used_index_page"])
    print("\n--- Raw text preview ---\n")
    print(sample["raw_text"][:1500])

Total filings stored: 65440

Sample doc: LOGC 8-K 2025-08-07 | used_index = False

--- Raw text preview ---

8-K
false
0002064307
0002064307
2025-08-07
2025-08-07
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
WASHINGTON, D.C. 20549
FORM
8-K
CURRENT REPORT
Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934
Date of Report (Date of earliest event reported):
August 07, 2025
ContextLogic Holdings Inc.
(Exact name of Registrant as Specified in Its Charter)
Delaware
333-286589
27-2930953
(State or Other Jurisdiction
of Incorporation)
(Commission File Number)
(IRS Employer
Identification No.)
2648 International Blvd., Ste 115
Oakland
,
California
94601
(Address of Principal Executive Offices)
(Zip Code)
Registrant’s Telephone Number, Including Area Code:
(415)
965-8476
N/A
(Former Name or Former Address, if Changed Since Last Report)
Check the appropriate box below if the Form 8-K filing is intended to simultaneously satisfy the filing obligation of the registrant under 